In [1]:

import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.models import Model

DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir   = os.path.join(DATASET_PATH, "val")
test_dir  = os.path.join(DATASET_PATH, "test")

CLASSES = ["edible", "poisonous"]

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FINE_TUNE = 5

PRUNE_RATIO = 0.30   # 🔥 30%

def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

attention_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_test_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

attention_data = attention_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

BASE_MODEL_PATH = r"D:\Project Jupyter\MODEL\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"


Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.
Found 2256 images belonging to 2 classes.


In [2]:
base_model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)

base_model.trainable = True
base_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 960)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               123008    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 258       
                                                                 
Total params: 3,119,618
Trainable params: 3,095,218
Non-

In [3]:
se_layers = [
    layer for layer in base_model.layers
    if "se" in layer.name.lower() and len(layer.output_shape) == 2
]

print("SE layers:")
for l in se_layers:
    print(l.name, l.output_shape)


SE layers:
dense (None, 128)
dense_1 (None, 2)


In [4]:
attention_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=[l.output for l in se_layers]
)


In [5]:
def compute_se_importance_per_block(att_model, data_gen, steps=10):
    block_importance = [None] * len(att_model.outputs)

    for _ in range(steps):
        x, _ = next(data_gen)
        se_outputs = att_model(x, training=False)

        for i, out in enumerate(se_outputs):
            out = tf.convert_to_tensor(out)  # (B, C)
            score = tf.reduce_mean(out, axis=0).numpy()

            if block_importance[i] is None:
                block_importance[i] = score
            else:
                block_importance[i] += score

    return [imp / steps for imp in block_importance]


In [6]:
se_importance_blocks = compute_se_importance_per_block(
    attention_model,
    attention_data,
    steps=20
)


In [7]:
prune_masks = []

for imp in se_importance_blocks:
    threshold = np.percentile(imp, PRUNE_RATIO * 100)
    mask = imp > threshold
    prune_masks.append(mask)

    print(f"Keep {np.sum(mask)} / {len(mask)} channels")


Keep 89 / 128 channels
Keep 1 / 2 channels


In [8]:
from tensorflow.keras.layers import Conv2D

conv_layers = [
    l for l in base_model.layers
    if isinstance(l, Conv2D)
]


In [9]:
for conv, mask in zip(conv_layers, prune_masks):
    weights = conv.get_weights()

    kernel = weights[0][:, :, :, mask]
    bias = weights[1][mask] if len(weights) > 1 else None

    conv.set_weights([kernel, bias] if bias is not None else [kernel])


In [10]:
import time

base_model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

history = base_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_FINE_TUNE
)

end_time = time.time()

training_time = end_time - start_time

print(f"\nTotal Training Time : {training_time:.2f} seconds")
print(f"Total Training Time : {training_time/60:.2f} minutes")
print(f"Avg Time / Epoch    : {training_time/EPOCHS_FINE_TUNE:.2f} seconds")


Epoch 1/5
71/71 [==============================] - 719s 10s/step - loss: 0.2860 - accuracy: 0.8825 - val_loss: 0.2377 - val_accuracy: 0.9113
Epoch 2/5
71/71 [==============================] - 652s 9s/step - loss: 0.1615 - accuracy: 0.9353 - val_loss: 0.1914 - val_accuracy: 0.9397
Epoch 3/5
71/71 [==============================] - 643s 9s/step - loss: 0.0950 - accuracy: 0.9707 - val_loss: 0.1607 - val_accuracy: 0.9468
Epoch 4/5
71/71 [==============================] - 655s 9s/step - loss: 0.0620 - accuracy: 0.9809 - val_loss: 0.1444 - val_accuracy: 0.9574
Epoch 5/5
71/71 [==============================] - 648s 9s/step - loss: 0.0503 - accuracy: 0.9863 - val_loss: 0.1303 - val_accuracy: 0.9468

Total Training Time : 3378.61 seconds
Total Training Time : 56.31 minutes
Avg Time / Epoch    : 675.72 seconds


In [11]:
PRUNED_MODEL_PATH = r"D:\Project Jupyter\mobilenetv3_se_attention_pruned_final30"

base_model.save(
    PRUNED_MODEL_PATH,
    include_optimizer=False
)


INFO:tensorflow:Assets written to: D:\Project Jupyter\mobilenetv3_se_attention_pruned_final30\assets


INFO:tensorflow:Assets written to: D:\Project Jupyter\mobilenetv3_se_attention_pruned_final30\assets


In [12]:
import os

def get_model_size_mb(model_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)


In [25]:
base_model_size = get_model_size_mb(BASE_MODEL_PATH)
pruned_model_size = get_model_size_mb(PRUNED_MODEL_PATH)

print(f"Ukuran Baseline Model : {base_model_size:.2f} MB")
print(f"Ukuran Pruned Model   : {pruned_model_size:.2f} MB")

compression_ratio = base_model_size / pruned_model_size
size_reduction = (1 - (pruned_model_size / base_model_size)) * 100

print(f"Compression Ratio : {compression_ratio:.2f}x")
print(f"Size Reduction    : {size_reduction:.2f}%")


Ukuran Baseline Model : 17.87 MB
Ukuran Pruned Model   : 17.08 MB
Compression Ratio : 1.05x
Size Reduction    : 4.37%


In [27]:
from sklearn.metrics import classification_report, confusion_matrix


In [29]:
test_data.reset()


In [31]:
base_model = tf.keras.models.load_model(
    PRUNED_MODEL_PATH,
    compile=False
)

predictions = base_model.predict(test_data, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes


8/8 [==============================] - 28s 3s/step


In [32]:
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASSES
))



Classification Report:
              precision    recall  f1-score   support

      edible       0.98      0.94      0.96       120
   poisonous       0.94      0.98      0.96       112

    accuracy                           0.96       232
   macro avg       0.96      0.96      0.96       232
weighted avg       0.96      0.96      0.96       232



In [33]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))



Confusion Matrix:
[[113   7]
 [  2 110]]


In [34]:
import time
import numpy as np

def benchmark_inference(model, data_gen, warmup=3, runs=10):
    """
    model      : final pruned model (tanpa pruning wrapper)
    data_gen   : test_data (shuffle=False)
    """

    # Ambil 1 batch
    x_batch, _ = next(data_gen)

    # Warm-up (penting)
    for _ in range(warmup):
        _ = model.predict(x_batch, verbose=0)

    times = []

    for _ in range(runs):
        start = time.time()
        _ = model.predict(x_batch, verbose=0)
        end = time.time()
        times.append(end - start)

    avg_time = np.mean(times)
    per_image_time = avg_time / x_batch.shape[0]

    print("\nInference Benchmark (CPU)")
    print(f"Batch size         : {x_batch.shape[0]}")
    print(f"Avg batch time     : {avg_time:.4f} seconds")
    print(f"Avg per image time : {per_image_time:.6f} seconds")

    return avg_time, per_image_time


In [35]:
benchmark_inference(base_model, test_data)



Inference Benchmark (CPU)
Batch size         : 32
Avg batch time     : 1.7791 seconds
Avg per image time : 0.055598 seconds


(1.7791447639465332, 0.05559827387332916)